In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("IMDB Dataset.csv")

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

In [8]:
# pre-processing
df["review"] = df["review"].str.lower()

In [9]:
import re

In [10]:
def remove_url(text):
    text = re.sub(r"http\S+"  ,"" , text)
    return text

df["review"] = df["review"].apply(remove_url)

In [11]:
def remove_punc(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "" , text)
    return text
df["review"] = df["review"].apply(remove_punc)

In [12]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "" , text)
    return text
df["review"] = df["review"].apply(remove_html)

In [13]:
import nltk

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words('english')

    for words in stop_words:
        if words in stop_words:
            text = text.replace(words , "")

    return text
df["review"] = df["review"].apply(remove_stopwords)

In [16]:
df.head()

,review,sentiment
0,ne f r r h enne h fer wchng 1 z epe u hke ...,positive
1,wnerful lle prucn br br flng echnque r unun...,positive
2,hugh w wnerful w pen e n h uer eken ng n...,positive
3,bc fl w lle b jke hnk zbe n cle pn fg...,negative
4,peer e l n e f ne vu unnng fl wch r e ffer...,positive


In [17]:
from nltk.stem import PorterStemmer

In [18]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"] = df["review"].apply(stemming)

In [19]:
df.head()

,review,sentiment
0,ne f r r h enn h fer wchng 1 z epe u hke rgh e...,positive
1,wner lle prucn br br flng echnqu r unung r leb...,positive
2,hugh w wner w pen e n h uer eken ng n r cnne e...,positive
3,bc fl w lle b jke hnk zbe n cle pn fghng ebr b...,negative
4,peer e l n e f ne vu unnng fl wch r e ffer u v...,positive


In [20]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [21]:
y = df["sentiment"]

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

In [23]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3523137 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3136)	0.07612514112330851
  (0, 1451)	0.060724553067064126
  (0, 1783)	0.045867176870267534
  (0, 4788)	0.11259579486542624
  (0, 1490)	0.12024514596199964
  (0, 2360)	0.10287832973878025
  (0, 4128)	0.11665098715022361
  (0, 1669)	0.08394320000658981
  (0, 4810)	0.20328110168238658
  (0, 2438)	0.08435359778286307
  (0, 1174)	0.07411828902090345
  (0, 345)	0.08743154936557365
  (0, 1945)	0.11497545004361671
  (0, 2395)	0.039693950816877356
  (0, 4300)	0.21061540899764988
  (0, 407)	0.03501225302531215
  (0, 398)	0.10461739932668719
  (0, 562)	0.04432725462980286
  (0, 4730)	0.34497218157033005
  (0, 4812)	0.043314000445069195
  (0, 4903)	0.043485312923360206
  (0, 4293)	0.07100240137700699
  (0, 1911)	0.039082651076863525
  (0, 4093)	0.05692206529698189
  (0, 3942)	0.05680732710816981
  :	:
  (49581, 1222)	0.12054209380548662
  (49581, 4912)	0.24248007966

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
X_train.shape

(39665, 5000)

In [26]:
X_test.shape

(9917, 5000)

In [27]:
import torch
from torch.utils.data import Dataset, DataLoader , TensorDataset

In [28]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [29]:
train_set = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train.values).float())

test_set = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test.values).float())

In [30]:
train_loader = DataLoader(train_set , shuffle = True, batch_size = 64)
test_loader = DataLoader(test_set , shuffle = True, batch_size = 64)

In [31]:
import torch.nn as nn
import torch.optim as optim

In [38]:
class RNN(nn.Module):
    def __init__(self , input_size, hidden_size = 128 , num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN Layer
        self.rnn = nn.RNN( input_size , hidden_size,num_layers , batch_first = True)
        self.fc = nn.Linear( hidden_size , 1)

    def forward(self , x):
        h0 = torch.zeros(self.num_layers , x.size(0) , self.hidden_size)

        out , _ = self.rnn(x, h0)
        out = self.fc(out[: , -1 , :])
        return out

In [41]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCEWithLogitsLoss()
opti = optim.Adam(model.parameters())


In [42]:
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for Xb, yb in train_loader:
        opti.zero_grad()

        # 1. Fixed typo: unsqueeze(1) adds a channel dimension
        Xb = Xb.unsqueeze(1)

        # 2. Forward pass
        logits = model(Xb).squeeze() # We call them 'logits' if they aren't activated yet

        # 3. Use raw logits with BCEWithLogitsLoss for better stability
        # Note: Ensure yb is a float for BCE loss functions
        loss = criterion(logits, yb.float())

        loss.backward()
        opti.step()

        running_loss += loss.item()

    # Calculate average loss for the whole epoch
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

Epoch 1/10 | Avg Loss: 0.4206
Epoch 2/10 | Avg Loss: 0.3115
Epoch 3/10 | Avg Loss: 0.2949
Epoch 4/10 | Avg Loss: 0.2874
Epoch 5/10 | Avg Loss: 0.2833
Epoch 6/10 | Avg Loss: 0.2792
Epoch 7/10 | Avg Loss: 0.2773
Epoch 8/10 | Avg Loss: 0.2756
Epoch 9/10 | Avg Loss: 0.2740
Epoch 10/10 | Avg Loss: 0.2729


In [44]:
model.eval()
with (torch.no_grad()):
    correct_vals = 0
    tot_vals = 0
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

print(f"accuracy = {correct_vals/tot_vals*100}%")

accuracy = 83.20056468690127%
